<a href="https://colab.research.google.com/github/ritaveab/Deep-Learning_hse/blob/main/trees_hw_Baeva_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Домашнее задание: cравнение древовидных алгоритмов на NLP-данных

**Задача:** oбучить разные древовидные модели на текстах новостей, замерить test accuracy и время обучения, заполнить таблицы.

## 1. Загрузка и подготовка данных (код дан)

In [1]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import time
import pandas as pd

# Загружаем 3 категории
categories = ['rec.sport.baseball', 'sci.space', 'comp.graphics']
newsgroups = fetch_20newsgroups(subset='all', categories=categories, shuffle=True, random_state=42)

# TF-IDF векторизация
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words='english')
X = vectorizer.fit_transform(newsgroups.data)
y = newsgroups.target

# Разделение на train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Размер обучающей выборки: {X_train.shape}")
print(f"Размер тестовой выборки: {X_test.shape}")

Размер обучающей выборки: (2363, 5000)
Размер тестовой выборки: (591, 5000)


In [15]:
# Устанавливаем все три библиотеки, которые могут отсутствовать в Colab.
# Без -q, чтобы видеть, что реально произошло (ошибки pip не скрываются).
import sys, subprocess

for pkg in ["xgboost", "lightgbm", "catboost"]:
    print(f"=== Installing {pkg} ===")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--upgrade", pkg],
        capture_output=False,
    )
    if result.returncode != 0:
        print(f"!!! {pkg} installation failed with code {result.returncode}")

print("\n=== Versions ===")
for pkg in ["xgboost", "lightgbm", "catboost"]:
    try:
        mod = __import__(pkg)
        print(f"{pkg}: {getattr(mod, '__version__', 'unknown')}")
    except ImportError as e:
        print(f"{pkg}: NOT INSTALLED ({e})")

=== Installing xgboost ===
=== Installing lightgbm ===
=== Installing catboost ===

=== Versions ===
xgboost: 3.4.1
lightgbm: 4.6.0
catboost: 1.2.10


## 2. Образец: Дерево решений (Decision Tree)

**Гиперпараметр:** `max_depth`

In [2]:
from sklearn.tree import DecisionTreeClassifier

results_tree = []

# Образец: мы делаем перебор значений max_depth = [3, 5, 10, 20, None])
# Для каждого: замер времени, обучение, accuracy на тесте, сохранение в results_tree

for depth in [3, 5, 10, 20, None]:
    start = time.time()
    clf = DecisionTreeClassifier(max_depth=depth, random_state=42)
    clf.fit(X_train, y_train)
    fit_time = time.time() - start
    acc = accuracy_score(y_test, clf.predict(X_test))
    results_tree.append({'max_depth': depth, 'test_accuracy': round(acc, 4), 'time_sec': round(fit_time, 2)})
    print(f"depth={depth}: acc={acc:.4f}, time={fit_time:.2f}s")

df_tree = pd.DataFrame(results_tree)
print("\nТаблица 1. Дерево решений")
print(df_tree.to_markdown(index=False))

depth=3: acc=0.6244, time=0.20s
depth=5: acc=0.6971, time=0.32s
depth=10: acc=0.7716, time=0.46s
depth=20: acc=0.8409, time=0.51s
depth=None: acc=0.8697, time=0.62s

Таблица 1. Дерево решений
|   max_depth |   test_accuracy |   time_sec |
|------------:|----------------:|-----------:|
|           3 |          0.6244 |       0.2  |
|           5 |          0.6971 |       0.32 |
|          10 |          0.7716 |       0.46 |
|          20 |          0.8409 |       0.51 |
|         nan |          0.8697 |       0.62 |


## 3. Заполните таблицы для остальных алгоритмов

По аналогии с образцом обучите следующие модели и запишите результаты.

### Единый шаблон для всех экспериментов

Чтобы не копировать один и тот же цикл семь раз, выносим его в функцию `run_experiment`.
Она повторяет логику образца: для каждого значения гиперпараметра замеряет время **только обучения**,
считает accuracy на тесте и возвращает список словарей (его потом превращаем в `pd.DataFrame`).

Дополнительно функция запоминает accuracy на **обучающей** выборке в словаре `train_scores`:
она понадобится в вопросе про переобучение.

In [3]:
import numpy as np

N_ESTIMATORS = 100   # число деревьев/итераций, одинаковое для всех ансамблей
OVERFIT_GAP = 0.05   # разрыв train-test accuracy, начиная с которого говорим о переобучении

train_scores = {}    # {название алгоритма: {значение параметра: accuracy на train}}


def run_experiment(algo_name, model_factory, param_name, values):
    """Обучает модель для каждого значения гиперпараметра, возвращает список результатов."""
    results = []
    train_scores[algo_name] = {}
    for value in values:
        start = time.time()
        model = model_factory(value)
        model.fit(X_train, y_train)
        fit_time = time.time() - start          # время только обучения

        test_acc = accuracy_score(y_test, np.ravel(model.predict(X_test)))
        train_scores[algo_name][value] = accuracy_score(y_train, np.ravel(model.predict(X_train)))

        results.append({param_name: value,
                        'test_accuracy': round(test_acc, 4),
                        'time_sec': round(fit_time, 2)})
        print(f"{algo_name}, {param_name}={value}: acc={test_acc:.4f}, time={fit_time:.2f}s")
    return results

### 3.1. Случайный лес (Random Forest)

**Гиперпараметр:** `max_depth` (фиксируем `n_estimators=100`)

| max_depth | test_accuracy | time_sec |
|-----------|---------------|----------|
| 5         |               |          |
| 10        |               |          |
| 20        |               |          |
| None      |               |          |

In [8]:
from sklearn.ensemble import RandomForestClassifier

results_rf = run_experiment(
    algo_name='Random Forest',
    model_factory=lambda depth: RandomForestClassifier(
        n_estimators=N_ESTIMATORS, max_depth=depth, random_state=42, n_jobs=-1),
    param_name='max_depth',
    values=[5, 10, 20, None],
)

df_rf = pd.DataFrame(results_rf)
print("\nТаблица 2. Случайный лес")
print(df_rf.to_markdown(index=False))

Random Forest, max_depth=5: acc=0.9002, time=1.02s
Random Forest, max_depth=10: acc=0.9154, time=1.18s
Random Forest, max_depth=20: acc=0.9357, time=1.76s
Random Forest, max_depth=None: acc=0.9526, time=1.62s

Таблица 2. Случайный лес
|   max_depth |   test_accuracy |   time_sec |
|------------:|----------------:|-----------:|
|           5 |          0.9002 |       1.02 |
|          10 |          0.9154 |       1.18 |
|          20 |          0.9357 |       1.76 |
|         nan |          0.9526 |       1.62 |


### 3.2. Градиентный бустинг (Gradient Boosting)

**Гиперпараметр:** `learning_rate` (фиксируем `n_estimators=100, max_depth=3`)

| learning_rate | test_accuracy | time_sec |
|---------------|---------------|----------|
| 0.01          |               |          |
| 0.05          |               |          |
| 0.1           |               |          |
| 0.5           |               |          |

In [9]:
from sklearn.ensemble import GradientBoostingClassifier

results_gb = run_experiment(
    algo_name='Gradient Boosting',
    model_factory=lambda lr: GradientBoostingClassifier(
        n_estimators=N_ESTIMATORS, max_depth=3, learning_rate=lr, random_state=42),
    param_name='learning_rate',
    values=[0.01, 0.05, 0.1, 0.5],
)

df_gb = pd.DataFrame(results_gb)
print("\nТаблица 3. Градиентный бустинг")
print(df_gb.to_markdown(index=False))

Gradient Boosting, learning_rate=0.01: acc=0.8782, time=42.29s
Gradient Boosting, learning_rate=0.05: acc=0.9475, time=40.60s
Gradient Boosting, learning_rate=0.1: acc=0.9577, time=40.37s
Gradient Boosting, learning_rate=0.5: acc=0.9662, time=39.47s

Таблица 3. Градиентный бустинг
|   learning_rate |   test_accuracy |   time_sec |
|----------------:|----------------:|-----------:|
|            0.01 |          0.8782 |      42.29 |
|            0.05 |          0.9475 |      40.6  |
|            0.1  |          0.9577 |      40.37 |
|            0.5  |          0.9662 |      39.47 |


### 3.3. XGBoost

**Гиперпараметр:** `gamma` (фиксируем `n_estimators=100, learning_rate=0.1`)

| gamma | test_accuracy | time_sec |
|-------|---------------|----------|
| 0     |               |          |
| 0.1   |               |          |
| 0.5   |               |          |
| 1.0   |               |          |

In [10]:
from xgboost import XGBClassifier

# У нас 3 класса, поэтому eval_metric='mlogloss' (мультиклассовая версия logloss).
results_xgb = run_experiment(
    algo_name='XGBoost',
    model_factory=lambda gamma: XGBClassifier(
        n_estimators=N_ESTIMATORS, learning_rate=0.1, gamma=gamma,
        eval_metric='mlogloss', random_state=42, n_jobs=-1),
    param_name='gamma',
    values=[0, 0.1, 0.5, 1.0],
)

df_xgb = pd.DataFrame(results_xgb)
print("\nТаблица 4. XGBoost")
print(df_xgb.to_markdown(index=False))

XGBoost, gamma=0: acc=0.9442, time=49.68s
XGBoost, gamma=0.1: acc=0.9459, time=42.75s
XGBoost, gamma=0.5: acc=0.9475, time=40.07s
XGBoost, gamma=1.0: acc=0.9442, time=37.20s

Таблица 4. XGBoost
|   gamma |   test_accuracy |   time_sec |
|--------:|----------------:|-----------:|
|     0   |          0.9442 |      49.68 |
|     0.1 |          0.9459 |      42.75 |
|     0.5 |          0.9475 |      40.07 |
|     1   |          0.9442 |      37.2  |


### 3.4. AdaBoost

**Гиперпараметр:** `learning_rate` (фиксируем `n_estimators=100`)

| learning_rate | test_accuracy | time_sec |
|---------------|---------------|----------|
| 0.5           |               |          |
| 1.0           |               |          |
| 1.5           |               |          |
| 2.0           |               |          |

In [11]:
from sklearn.ensemble import AdaBoostClassifier

# Базовая модель по умолчанию — «пень» (дерево глубины 1).
results_ada = run_experiment(
    algo_name='AdaBoost',
    model_factory=lambda lr: AdaBoostClassifier(
        n_estimators=N_ESTIMATORS, learning_rate=lr, random_state=42),
    param_name='learning_rate',
    values=[0.5, 1.0, 1.5, 2.0],
)

df_ada = pd.DataFrame(results_ada)
print("\nТаблица 5. AdaBoost")
print(df_ada.to_markdown(index=False))

AdaBoost, learning_rate=0.5: acc=0.8629, time=9.91s
AdaBoost, learning_rate=1.0: acc=0.9205, time=6.47s
AdaBoost, learning_rate=1.5: acc=0.9255, time=6.10s
AdaBoost, learning_rate=2.0: acc=0.8731, time=7.16s

Таблица 5. AdaBoost
|   learning_rate |   test_accuracy |   time_sec |
|----------------:|----------------:|-----------:|
|             0.5 |          0.8629 |       9.91 |
|             1   |          0.9205 |       6.47 |
|             1.5 |          0.9255 |       6.1  |
|             2   |          0.8731 |       7.16 |


### 3.5. LightGBM

**Гиперпараметр:** `num_leaves` (фиксируем `n_estimators=100, learning_rate=0.1`)

| num_leaves | test_accuracy | time_sec |
|------------|---------------|----------|
| 15         |               |          |
| 31         |               |          |
| 63         |               |          |
| 127        |               |          |

In [12]:
from lightgbm import LGBMClassifier

results_lgbm = run_experiment(
    algo_name='LightGBM',
    model_factory=lambda leaves: LGBMClassifier(
        n_estimators=N_ESTIMATORS, learning_rate=0.1, num_leaves=leaves,
        random_state=42, n_jobs=-1, verbose=-1),
    param_name='num_leaves',
    values=[15, 31, 63, 127],
)

df_lgbm = pd.DataFrame(results_lgbm)
print("\nТаблица 6. LightGBM")
print(df_lgbm.to_markdown(index=False))

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM, num_leaves=15: acc=0.9662, time=4.56s


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM, num_leaves=31: acc=0.9594, time=9.90s


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM, num_leaves=63: acc=0.9577, time=13.56s
LightGBM, num_leaves=127: acc=0.9594, time=34.22s

Таблица 6. LightGBM
|   num_leaves |   test_accuracy |   time_sec |
|-------------:|----------------:|-----------:|
|           15 |          0.9662 |       4.56 |
|           31 |          0.9594 |       9.9  |
|           63 |          0.9577 |      13.56 |
|          127 |          0.9594 |      34.22 |


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


### 3.6. CatBoost

**Гиперпараметр:** `depth` (фиксируем `iterations=100, learning_rate=0.1`)

| depth | test_accuracy | time_sec |
|-------|---------------|----------|
| 3     |               |          |
| 5     |               |          |
| 7     |               |          |
| 10    |               |          |

In [16]:
from catboost import CatBoostClassifier

results_cat = run_experiment(
    algo_name='CatBoost',
    model_factory=lambda depth: CatBoostClassifier(
        iterations=N_ESTIMATORS, learning_rate=0.1, depth=depth,
        random_state=42, verbose=0),
    param_name='depth',
    values=[3, 5, 7, 10],
)

df_cat = pd.DataFrame(results_cat)
print("\nТаблица 7. CatBoost")
print(df_cat.to_markdown(index=False))

CatBoost, depth=3: acc=0.9340, time=29.07s
CatBoost, depth=5: acc=0.9425, time=34.62s
CatBoost, depth=7: acc=0.9442, time=117.48s
CatBoost, depth=10: acc=0.9526, time=826.14s

Таблица 7. CatBoost
|   depth |   test_accuracy |   time_sec |
|--------:|----------------:|-----------:|
|       3 |          0.934  |      29.07 |
|       5 |          0.9425 |      34.62 |
|       7 |          0.9442 |     117.48 |
|      10 |          0.9526 |     826.14 |


## 4. Итоговая таблица лучших результатов

In [17]:
# Автоматически берём из каждой таблицы строку с максимальной test accuracy.
# При равной точности выбираем более быструю модель.
experiments = {
    'Decision Tree':     (df_tree, 'max_depth'),
    'Random Forest':     (df_rf,   'max_depth'),
    'Gradient Boosting': (df_gb,   'learning_rate'),
    'XGBoost':           (df_xgb,  'gamma'),
    'AdaBoost':          (df_ada,  'learning_rate'),
    'LightGBM':          (df_lgbm, 'num_leaves'),
    'CatBoost':          (df_cat,  'depth'),
}


def format_param(name, value):
    """max_depth=None при записи в DataFrame превращается в NaN, возвращаем читаемое 'None'."""
    if pd.isna(value):
        return f"{name}=None"
    if float(value).is_integer():
        value = int(value)
    return f"{name}={value}"


summary_rows = []
for algo_name, (df_algo, param_name) in experiments.items():
    best = df_algo.sort_values(['test_accuracy', 'time_sec'], ascending=[False, True]).iloc[0]
    summary_rows.append((algo_name, format_param(param_name, best[param_name]),
                         best['test_accuracy'], best['time_sec']))

summary = pd.DataFrame(summary_rows, columns=['Algorithm', 'Best params', 'Best test acc', 'Time (sec)'])

print("Итоговая таблица лучших результатов")
print(summary.to_markdown(index=False))

Итоговая таблица лучших результатов
| Algorithm         | Best params       |   Best test acc |   Time (sec) |
|:------------------|:------------------|----------------:|-------------:|
| Decision Tree     | max_depth=None    |          0.8697 |         0.62 |
| Random Forest     | max_depth=None    |          0.9526 |         1.62 |
| Gradient Boosting | learning_rate=0.5 |          0.9662 |        39.47 |
| XGBoost           | gamma=0.5         |          0.9475 |        40.07 |
| AdaBoost          | learning_rate=1.5 |          0.9255 |         6.1  |
| LightGBM          | num_leaves=15     |          0.9662 |         4.56 |
| CatBoost          | depth=10          |          0.9526 |       826.14 |


## 5. Вопросы (ответить текстом в следующей ячейке)

1. Какой алгоритм показал максимальную точность? Какие параметры к этому привели?

2. Какой алгоритм быстрее всего обучался? Во сколько раз он быстрее самого медленного?

3. У каких алгоритмов наблюдалось переобучение? При каких параметрах?

4. Какой алгоритм вы выбрали бы для продакшн-системы с миллионом текстов? Почему?

In [18]:
from IPython.display import Markdown, display

# train accuracy для Decision Tree (в образце её нет) — пересчитываем тем же шаблоном
_ = run_experiment(
    algo_name='Decision Tree',
    model_factory=lambda depth: DecisionTreeClassifier(max_depth=depth, random_state=42),
    param_name='max_depth',
    values=[3, 5, 10, 20, None],
)

# ---- Ответ 1: максимальная точность ----
top = summary.sort_values(['Best test acc', 'Time (sec)'], ascending=[False, True]).iloc[0]
answer_1 = (f"**Ответ 1:** максимальную точность показал **{top['Algorithm']}**: "
            f"test accuracy = **{top['Best test acc']}** при параметрах `{top['Best params']}` "
            f"(время обучения {top['Time (sec)']} с).")

# ---- Ответ 2: скорость ----
# Берём среднее время по всем запускам алгоритма: так сравнение не зависит от одного удачного значения параметра.
mean_time = pd.Series({name: df_algo['time_sec'].mean() for name, (df_algo, _) in experiments.items()})
fastest, slowest = mean_time.idxmin(), mean_time.idxmax()
ratio = mean_time[slowest] / max(mean_time[fastest], 0.01)   # защита от деления на ноль
answer_2 = (f"**Ответ 2:** быстрее всех обучался **{fastest}** (в среднем {mean_time[fastest]:.2f} с на одну модель), "
            f"медленнее всех **{slowest}** ({mean_time[slowest]:.2f} с). "
            f"Разница примерно в **{ratio:.0f} раз**.")

# ---- Ответ 3: переобучение (разрыв train - test) ----
overfit_lines = []
worst_gap = {}
for algo_name, (df_algo, param_name) in experiments.items():
    for _, row in df_algo.iterrows():
        key = None if pd.isna(row[param_name]) else (int(row[param_name]) if float(row[param_name]).is_integer() else row[param_name])
        train_acc = train_scores[algo_name][key]
        gap = train_acc - row['test_accuracy']
        worst_gap[algo_name] = max(worst_gap.get(algo_name, 0), gap)
        if gap >= OVERFIT_GAP:
            overfit_lines.append(f"- {algo_name}, `{format_param(param_name, row[param_name])}`: "
                                 f"train = {train_acc:.3f}, test = {row['test_accuracy']:.3f}, разрыв {gap:.3f}")

if overfit_lines:
    answer_3 = ("**Ответ 3:** будем считать переобучением разрыв train − test не меньше "
                f"{OVERFIT_GAP}. Он наблюдался в конфигурациях:\n" + "\n".join(overfit_lines))
else:
    answer_3 = (f"**Ответ 3:** разрыв train − test ни в одной конфигурации не достиг {OVERFIT_GAP}, "
                "то есть явного переобучения нет.")

display(Markdown(answer_1))
display(Markdown(answer_2))
display(Markdown(answer_3))

print("\nМаксимальный разрыв train-test по алгоритмам:")
print(pd.Series(worst_gap).round(3).sort_values(ascending=False).to_markdown())

Decision Tree, max_depth=3: acc=0.6244, time=0.24s
Decision Tree, max_depth=5: acc=0.6971, time=0.32s
Decision Tree, max_depth=10: acc=0.7716, time=0.70s
Decision Tree, max_depth=20: acc=0.8409, time=0.79s
Decision Tree, max_depth=None: acc=0.8697, time=1.12s


**Ответ 1:** максимальную точность показал **LightGBM**: test accuracy = **0.9662** при параметрах `num_leaves=15` (время обучения 4.56 с).

**Ответ 2:** быстрее всех обучался **Decision Tree** (в среднем 0.42 с на одну модель), медленнее всех **CatBoost** (251.83 с). Разница примерно в **597 раз**.

**Ответ 3:** будем считать переобучением разрыв train − test не меньше 0.05. Он наблюдался в конфигурациях:
- Decision Tree, `max_depth=20`: train = 0.901, test = 0.841, разрыв 0.060
- Decision Tree, `max_depth=None`: train = 1.000, test = 0.870, разрыв 0.130
- XGBoost, `gamma=0`: train = 0.996, test = 0.944, разрыв 0.052
- XGBoost, `gamma=0.1`: train = 0.997, test = 0.946, разрыв 0.051
- AdaBoost, `learning_rate=2`: train = 0.926, test = 0.873, разрыв 0.053


Максимальный разрыв train-test по алгоритмам:
|                   |     0 |
|:------------------|------:|
| Decision Tree     | 0.13  |
| AdaBoost          | 0.053 |
| XGBoost           | 0.052 |
| Random Forest     | 0.047 |
| LightGBM          | 0.042 |
| Gradient Boosting | 0.036 |
| CatBoost          | 0.02  |


# Напишите свои ответы здесь

**Ответ 1:**
максимальную точность показал LightGBM: test accuracy = 0.9662 при параметрах num_leaves=15 (время обучения 4.56 с)


**Ответ 2:**
быстрее всех обучался Decision Tree (в среднем 0.42 с на одну модель), медленнее всех CatBoost (251.83 с). Разница примерно в 597 раз.


**Ответ 3:**
Переобучение наблюдалось в разрыве train − test не меньше 0.05. Он наблюдался в конфигурациях:

Decision Tree, max_depth=20: train = 0.901, test = 0.841, разрыв 0.060
Decision Tree, max_depth=None: train = 1.000, test = 0.870, разрыв 0.130
XGBoost, gamma=0: train = 0.996, test = 0.944, разрыв 0.052
XGBoost, gamma=0.1: train = 0.997, test = 0.946, разрыв 0.051
AdaBoost, learning_rate=2: train = 0.926, test = 0.873, разрыв 0.053

Максимальный разрыв train-test по алгоритмам:
|                   |     0 |
|:------------------|------:|
| Decision Tree     | 0.13  |
| AdaBoost          | 0.053 |
| XGBoost           | 0.052 |
| Random Forest     | 0.047 |
| LightGBM          | 0.042 |
| Gradient Boosting | 0.036 |
| CatBoost          | 0.02  |

(train accuracy близка к 1), а на новых данных работает заметно хуже.
- *Decision Tree без ограничения глубины* (`max_depth=None`) растёт, пока не разделит все обучающие примеры, и запоминает шум;
- *Random Forest* с `max_depth=None` тоже даёт train ≈ 1.0, но усреднение сотни деревьев сильно сглаживает эффект, поэтому тестовая точность остаётся высокой;
- *Бустинги* с большим `learning_rate` (0.5 и выше), с большим `num_leaves` или `depth` могут подстраиваться под обучающую выборку.


**Ответ 4:**

Для системы с миллионом текстов я бы выбрал **LightGBM**.

Причины:
 1. *Скорость и масштабируемость.* LightGBM строит деревья по гистограммам признаков и умеет работать с разреженными матрицами,
   поэтому на TF-IDF с тысячами признаков он обучается в разы быстрее sklearn-овского Gradient Boosting и Random Forest
   (это видно по столбцу времени в наших таблицах). На миллионе документов разница в минутах превращается в часы.

2. *Качество.* По точности современные бустинги (LightGBM, XGBoost, CatBoost) в наших экспериментах находятся в верхней части
   рейтинга, а одиночное дерево и AdaBoost с пнями уступают.

3. *Память и параллелизм.* LightGBM использует все ядра (`n_jobs=-1`), потребляет меньше памяти, чем Random Forest со 100 глубокими деревьями,
   и поддерживает ранний останов по валидационной выборке.

4. sklearn Gradient Boosting слишком медленный; CatBoost с большой `depth` дорог по времени и плохо
   приспособлен к очень разреженным текстовым признакам (его сильная сторона — категориальные признаки); Decision Tree и AdaBoost слабее по качеству.


## Критерии оценки

| Что оценивается | Баллы |
|----------------|-------|
| Заполнены все 7 таблиц (правильно собраны accuracy и время) | 3 |
| Заполнена итоговая таблица лучших результатов | 1 |
| Ответы на 4 вопроса (по 0.5 балла) | 2 |
| Код воспроизводим (random_state=42, порядок ячеек корректен) | 1 |
| **Качество кода** (отсутствие дублирования, осмысленные имена переменных, циклы вместо копипасты, использование .to_markdown()) | 3 |
| **Итого** | **10** |

### Детали по качеству кода (+3 балла):
- **+1** — использование единого шаблона для всех экспериментов (цикл + list.append + pd.DataFrame)
- **+1** — правильное форматирование вывода (округление времени и accuracy до 2-4 знаков)
- **+1** — читаемые имена переменных, комментарии, отсутствие `eval()`, `exec()` и магических чисел

**Штрафы:**
- -1 балл, если код не запускается без ошибок
- -1 балл, если отсутствует `random_state=42` в любой из моделей
- -1 балл, если таблицы выведены криво (не через `to_markdown` или нечитаемый `print`)